<a href="https://colab.research.google.com/github/pepealania/agentic-rag/blob/main/PoC/POC3_Qwen3_8B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q openai pydantic pandas

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
LangGraph: 1.2.9


In [ ]:
import os
import json
import time
import pandas as pd
from openai import OpenAI
from pydantic import BaseModel, Field

MODEL_NAME = "qwen3:8b"
BASE_URL = "http://localhost:11434/v1"
TEMPERATURE = 0.0

client = OpenAI(
    base_url=BASE_URL,
    api_key="ollama"
)

print("Model:", MODEL_NAME)
print("Base URL:", BASE_URL)


>>> Nodo 1: recuperar
>>> Nodo 2: analizar

========== RESULTADO ==========
Consulta: ¿Qué framework permite controlar un workflow agentic?
Evidencias: 2
Conclusión: las evidencias son suficientes.

========== HISTORIAL ==========

--- Snapshot 0 ---
Step: 2
Next: ()
Values: {'consulta': '¿Qué framework permite controlar un workflow agentic?', 'evidencias': ['Documento A: LangGraph permite construir workflows con nodos y aristas.', 'Documento B: El estado puede compartirse entre los nodos.'], 'resultado': 'Consulta: ¿Qué framework permite controlar un workflow agentic?\nEvidencias: 2\nConclusión: las evidencias son suficientes.'}

--- Snapshot 1 ---
Step: 1
Next: ('analizar',)
Values: {'consulta': '¿Qué framework permite controlar un workflow agentic?', 'evidencias': ['Documento A: LangGraph permite construir workflows con nodos y aristas.', 'Documento B: El estado puede compartirse entre los nodos.'], 'resultado': ''}

--- Snapshot 2 ---
Step: 0
Next: ('recuperar',)
Values: {'consulta

In [ ]:
class Citation(BaseModel):
    document_id: str
    chunk_id: str


class Claim(BaseModel):
    text: str
    citations: list[Citation]


class LLMResponse(BaseModel):
    answer: str
    claims: list[Claim]
    abstained: bool


In [ ]:
TEST_CASES = [
    {
        "question_id": "Q01",
        "question": "¿Qué tareas asignadas a EMP-001 fueron completadas durante los seis meses?",
        "reference_answer": "EMP-001 completó las tareas TASK-001 a TASK-006.",
        "evidence": [
            {
                "document_id": "DOC-TASK-001",
                "chunk_id": "DOC-TASK-001-01",
                "content": "TASK-001 fue completada por EMP-001."
            },
            {
                "document_id": "DOC-TASK-002",
                "chunk_id": "DOC-TASK-002-01",
                "content": "TASK-002 fue completada por EMP-001."
            },
            {
                "document_id": "DOC-TASK-003",
                "chunk_id": "DOC-TASK-003-01",
                "content": "TASK-003 fue completada por EMP-001."
            },
            {
                "document_id": "DOC-TASK-004",
                "chunk_id": "DOC-TASK-004-01",
                "content": "TASK-004 fue completada por EMP-001."
            },
            {
                "document_id": "DOC-TASK-005",
                "chunk_id": "DOC-TASK-005-01",
                "content": "TASK-005 fue completada por EMP-001."
            },
            {
                "document_id": "DOC-TASK-006",
                "chunk_id": "DOC-TASK-006-01",
                "content": "TASK-006 fue completada por EMP-001."
            }
        ]
    },

    {
        "question_id": "Q10",
        "question": "¿Puede afirmarse que la sobrecarga explica todos los retrasos de EMP-010?",
        "reference_answer": "No. La sobrecarga está documentada solo para algunos retrasos.",
        "evidence": [
            {
                "document_id": "DOC-EMP010-M03",
                "chunk_id": "DOC-EMP010-M03-01",
                "content": "Durante M03 aumentó la carga de trabajo y se registró retraso en TASK-061."
            },
            {
                "document_id": "DOC-EMP010-M04",
                "chunk_id": "DOC-EMP010-M04-01",
                "content": "Durante M04 se registró retraso en TASK-062, pero no se documentó aumento de carga."
            }
        ]
    },

    {
        "question_id": "Q11",
        "question": "¿Existen documentos contradictorios sobre el cumplimiento de EMP-011 en M03?",
        "reference_answer": "Sí. Un reporte registra cumplimiento y una retroalimentación registra incumplimiento parcial.",
        "evidence": [
            {
                "document_id": "DOC-EMP011-M03-REPORT",
                "chunk_id": "DOC-EMP011-M03-REPORT-01",
                "content": "El reporte de avance registra cumplimiento de la tarea."
            },
            {
                "document_id": "DOC-EMP011-M03-FEEDBACK",
                "chunk_id": "DOC-EMP011-M03-FEEDBACK-01",
                "content": "La retroalimentación registra incumplimiento parcial de la tarea."
            }
        ]
    },

    {
        "question_id": "Q16",
        "question": "¿Qué porcentaje de las tareas de EMP-016 fueron consideradas excelentes por sus compañeros?",
        "reference_answer": "No respondible: no existe evidencia para calcular ese porcentaje.",
        "evidence": [
            {
                "document_id": "DOC-EMP016-M01",
                "chunk_id": "DOC-EMP016-M01-01",
                "content": "Se registran tareas y avances de EMP-016. No se incluyen evaluaciones de compañeros."
            }
        ]
    },

    {
        "question_id": "Q20",
        "question": "¿Cómo evolucionó el cumplimiento de tareas de EMP-020 entre M01 y M06?",
        "reference_answer": "Mejoró: presentó retrasos iniciales y cumplimiento estable desde M04.",
        "evidence": [
            {
                "document_id": "DOC-EMP020-M01",
                "chunk_id": "DOC-EMP020-M01-01",
                "content": "Durante M01 se registraron retrasos en tareas asignadas."
            },
            {
                "document_id": "DOC-EMP020-M03",
                "chunk_id": "DOC-EMP020-M03-01",
                "content": "Durante M03 todavía se registraron retrasos."
            },
            {
                "document_id": "DOC-EMP020-M04",
                "chunk_id": "DOC-EMP020-M04-01",
                "content": "Desde M04 las tareas fueron completadas dentro del plazo."
            },
            {
                "document_id": "DOC-EMP020-M06",
                "chunk_id": "DOC-EMP020-M06-01",
                "content": "Durante M06 se mantuvo el cumplimiento estable."
            }
        ]
    }
]
